In [ ]:
"""
Figure: Methods Pipeline (swim-lane diagram)
============================================

Renders the 21-phase ComputationalReview pipeline as a swim-lane diagram,
faithfully reproducing the actor/critic/validator separation defined in
skills/comprev-orchestrator-v29.md (Phase Index).

Lanes (top to bottom):
  - Coordinator   -- routing across phases; gates between phases (dotted rail)
  - DATAML agent  -- mechanical/programmatic actors and validators
  - LITREVIEW agent -- scientific judgment actors and critics

Element types (per Phase Index in comprev-orchestrator-v29.md):
  - ACTOR boxes    -- solid coloured by lane (DATAML blue / LITREVIEW orange)
  - CRITIC boxes   -- LITREVIEW phases with information barrier from the actor;
                      bordered in purple and hatched. Phases 6, 8, 12, 16.
  - VALIDATOR pins -- DATAML phases that re-check an actor's output; rendered
                      as smaller lighter-blue boxes pinned under their actor.
                      Phases 1V, 2V, 3V, 5V, 7V, 9V, 14V, 15V, 17V, 19V, 20V,
                      and Phase 21 (which is itself a deploy-polish validator
                      whose actor and validator are the same frame).

Special cases captured:
  - Phase 1 has TWO actors (LITREVIEW for scoping + DATAML for materialisation);
    both render at x=1, one per lane.
  - Phase 20a (Methods Ledger Refresh) is a DATAML actor that runs immediately
    before the Phase 20 push.
  - Phase 21 has no separate actor (the deployed site is the input); its box
    is rendered in the DATAML lane with a dashed validator border.

Layout: x-coordinates are sequential slot indices, not phase numbers. The
displayed phase number is the 6th element of each PHASES tuple.

Output: figures/fig_methods_pipeline.png
"""


In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch

# ---------------------------------------------------------------------------
# Authoritative phase table (per skills/comprev-orchestrator-v29.md Phase Index)
# ---------------------------------------------------------------------------

# (x_slot, label, role, kind, blinded, displayed_phase_number)
# role: "dataml" | "expert" (LITREVIEW)
# kind: "actor" | "critic" | "validator"
PHASES = [
    ( 1, "Scope",                  "expert", "actor",     False, "1"),
    ( 1, "Materialise",            "dataml", "actor",     False, "1"),
    ( 2, "Evidence\nGathering",    "expert", "actor",     False, "2"),
    ( 3, "Citation\nInfra",        "dataml", "actor",     False, "3"),
    ( 4, "Scaffold",               "expert", "actor",     False, "4"),
    ( 5, "Evidence\nCuration",     "dataml", "actor",     False, "5"),
    ( 6, "Figure\nAudit",          "expert", "critic",    True , "6"),
    ( 7, "Section\nDrafting",      "expert", "actor",     False, "7"),
    ( 8, "Section\nCritics",       "expert", "critic",    True , "8"),
    ( 9, "Bibliography",           "dataml", "actor",     False, "9"),
    (10, "Integration",            "expert", "actor",     False, "10"),
    (11, "Intro /\nConclusion",    "expert", "actor",     False, "11"),
    (12, "Bookend\nCritic",        "expert", "critic",    True , "12"),
    (13, "Methods",                "dataml", "actor",     False, "13"),
    (14, "Document\nAssembly",     "dataml", "actor",     False, "14"),
    (15, "Citation\nTriples",      "dataml", "actor",     False, "15"),
    (16, "Citation\nVerification", "expert", "critic",    True , "16"),
    (17, "Fix\nPreparation",       "dataml", "actor",     False, "17"),
    (18, "Fix\nExecution",         "expert", "actor",     False, "18"),
    (19, "Fix\nApplication",       "dataml", "actor",     False, "19"),
    (20, "Methods\nLedger\nRefresh","dataml","actor",     False, "20a"),
    (21, "Repository\nPush",       "dataml", "actor",     False, "20"),
    (22, "Deploy\nPolish",         "dataml", "validator", False, "21"),
]

# x-slot of each DATAML validator pin.  Each pin sits directly beneath its
# parent actor's x-position.  The label is "NV" where N is the *displayed*
# phase number of the parent actor.
PIN_LOCATIONS = [
    ( 1, "1V" ),    # under Phase 1 DATAML materialise actor
    ( 2, "2V" ),    # under Phase 2 LITREVIEW evidence actor
    ( 3, "3V" ),
    ( 5, "5V" ),
    ( 7, "7V" ),
    ( 9, "9V" ),
    (14, "14V"),
    (15, "15V"),
    (17, "17V"),
    (19, "19V"),
    (21, "20V"),    # under Phase 20 Repository Push actor
]

# ---------------------------------------------------------------------------
# Style
# ---------------------------------------------------------------------------
EDGE = {"coord": "#2e7d32", "dataml": "#1565c0", "expert": "#e65100"}
FILL = {"coord": "#c8e6c9", "dataml": "#bbdefb", "expert": "#ffe0b2"}
VALIDATOR_FILL = "#e3f2fd"
CRITIC_EDGE = "#6a1b9a"
CRITIC_HATCH = "////"
VAL_LINESTYLE = (0, (4, 2))

LANE_Y = {"coord": 4.0, "dataml": 2.1, "expert": 0.2}
LANE_LABEL = {
    "coord":  "COORDINATOR\n(routing + gates)",
    "dataml": "DATAML AGENT\n(mechanical actors\n+ validators)",
    "expert": "LITREVIEW AGENT\n(scientific actors\n+ blinded critics)",
}

# ---------------------------------------------------------------------------
# Figure
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(22, 7.2))
ax.set_xlim(-2.4, 24.0)
ax.set_ylim(-1.6, 5.7)
ax.axis("off")

# Lane backgrounds
for role, y in LANE_Y.items():
    ax.add_patch(FancyBboxPatch(
        (-1.9, y - 0.75), 25.6, 1.50,
        boxstyle="round,pad=0.02",
        facecolor=FILL[role], edgecolor="none", alpha=0.20, zorder=0,
    ))
    ax.text(-1.8, y, LANE_LABEL[role],
            ha="left", va="center", fontsize=8.6, weight="bold",
            color=EDGE[role], zorder=1)

# Coordinator gate rail (dotted)
GATE_Y = LANE_Y["coord"] - 0.82
ax.plot([0.5, 22.5], [GATE_Y, GATE_Y], color="#8a8a8a",
        linewidth=0.7, linestyle=(0, (1, 2)), zorder=1)
ax.text(22.7, GATE_Y, "  gate", ha="left", va="center", fontsize=7,
        color="#666", style="italic")

# ---------------------------------------------------------------------------
# Inter-phase arrows
# ---------------------------------------------------------------------------
box_w, box_h = 0.93, 0.92

# Build sequential connection list using primary lane for each phase.
# Phase 1 is represented by its LITREVIEW actor for routing.
primary_lane = {1: "expert", 2: "expert", 3: "dataml", 4: "expert",
                5: "dataml", 6: "expert", 7: "expert", 8: "expert",
                9: "dataml", 10: "expert", 11: "expert", 12: "expert",
                13: "dataml", 14: "dataml", 15: "dataml", 16: "expert",
                17: "dataml", 18: "expert", 19: "dataml", 20: "dataml",
                21: "dataml", 22: "dataml"}
slots = sorted(primary_lane)
for i in range(len(slots) - 1):
    x1, x2 = slots[i], slots[i + 1]
    y1, y2 = LANE_Y[primary_lane[x1]], LANE_Y[primary_lane[x2]]
    rad = 0.0 if primary_lane[x1] == primary_lane[x2] else \
          (-0.20 if y2 < y1 else 0.20)
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle="->", lw=0.9, color="#888",
                                connectionstyle=f"arc3,rad={rad}",
                                shrinkA=20, shrinkB=20),
                zorder=2)

# Vertical link Phase 1 LITREVIEW <-> Phase 1 DATAML materialise (same x)
ax.annotate("", xy=(1, LANE_Y["expert"] + box_h/2),
            xytext=(1, LANE_Y["dataml"] - box_h/2),
            arrowprops=dict(arrowstyle="-", lw=0.7, color="#aaa",
                            linestyle=(0, (2, 2))),
            zorder=2)

# ---------------------------------------------------------------------------
# Phase boxes
# ---------------------------------------------------------------------------
for x, label, role, kind, blinded, displayed_num in PHASES:
    y = LANE_Y[role]
    if kind == "critic":
        ec = CRITIC_EDGE
        lw = 1.8
        hatch = CRITIC_HATCH
        linestyle = VAL_LINESTYLE   # dashed — same convention as validator
        weight = "bold"
        fc = FILL[role]
    elif kind == "validator":
        ec = EDGE[role]
        lw = 1.6
        hatch = CRITIC_HATCH        # same hatch convention as critic — both are info-barriered reviewers
        linestyle = VAL_LINESTYLE
        weight = "regular"
        fc = VALIDATOR_FILL
    else:
        ec = EDGE[role]
        lw = 1.5
        hatch = None
        linestyle = "-"
        weight = "regular"
        fc = FILL[role]

    ax.add_patch(FancyBboxPatch(
        (x - box_w / 2, y - box_h / 2), box_w, box_h,
        boxstyle="round,pad=0.03",
        facecolor=fc, edgecolor=ec, linewidth=lw,
        hatch=hatch, linestyle=linestyle,
        zorder=3,
    ))
    ax.text(x, y, label, ha="center", va="center", fontsize=8.0,
            zorder=4, weight=weight)

    # Phase number label above box
    ax.text(x, y + box_h / 2 + 0.08, displayed_num,
            ha="center", va="bottom", fontsize=8.5, weight="bold",
            color=EDGE[role], zorder=4)

    # Coordinator-to-phase pin line (only for the primary lane to avoid
    # double-drawing for Phase 1's two actors)
    if not (x == 1 and role == "dataml"):
        ax.plot([x, x], [LANE_Y["coord"] + box_h / 2 - 0.5, GATE_Y],
                color="#bbb", linewidth=0.5, linestyle=":", zorder=1)

# ---------------------------------------------------------------------------
# Validator pins (small dashed-border boxes in the DATAML lane)
# ---------------------------------------------------------------------------
PIN_W, PIN_H = 0.62, 0.36
PIN_Y = LANE_Y["dataml"] - 0.82

for x, vlabel in PIN_LOCATIONS:
    ax.add_patch(FancyBboxPatch(
        (x - PIN_W / 2, PIN_Y - PIN_H / 2), PIN_W, PIN_H,
        boxstyle="round,pad=0.02",
        facecolor=VALIDATOR_FILL, edgecolor=EDGE["dataml"],
        linewidth=1.0, linestyle=VAL_LINESTYLE, hatch=CRITIC_HATCH,
        zorder=3,
    ))
    ax.text(x, PIN_Y, vlabel,
            ha="center", va="center", fontsize=7.5,
            color=EDGE["dataml"], weight="bold", zorder=4)
    # Connector pin to its parent actor (DATAML actor if present, else
    # to the LITREVIEW actor on the lane above)
    parent_lane = "dataml"
    # Phase 2V is the only validator whose actor is LITREVIEW
    if vlabel == "2V" or vlabel == "7V":
        parent_lane = "expert"
        ax.plot([x, x],
                [LANE_Y["expert"] - box_h / 2, PIN_Y + PIN_H / 2],
                color="#9e9e9e", linewidth=0.6, linestyle=":", zorder=1)
    else:
        ax.plot([x, x],
                [LANE_Y["dataml"] - box_h / 2, PIN_Y + PIN_H / 2],
                color="#9e9e9e", linewidth=0.6, linestyle=":", zorder=1)

# ---------------------------------------------------------------------------
# Footer caveat (below LITREVIEW lane)
# ---------------------------------------------------------------------------
ax.text(0.5, -1.05,
        "Phase 1 has two simultaneous actors (LITREVIEW scoping + DATAML materialisation, dashed link).  "
        "Phase 20a (Methods Ledger Refresh) runs between Phases 19V and 20.  "
        "Phase 21 (Deploy Polish) has no separate actor -- its actor and validator are the same frame, "
        "reading the deployed Pages artifact.",
        ha="left", va="top", fontsize=7.2, color="#555", style="italic",
        wrap=True)

# ---------------------------------------------------------------------------
# Legend
# ---------------------------------------------------------------------------
legend_items = [
    mpatches.Patch(facecolor=FILL["coord"], edgecolor=EDGE["coord"],
                   label="Coordinator -- routing + gates"),
    mpatches.Patch(facecolor=FILL["dataml"], edgecolor=EDGE["dataml"],
                   label="DATAML actor -- mechanical phase"),
    mpatches.Patch(facecolor=FILL["expert"], edgecolor=EDGE["expert"],
                   label="LITREVIEW actor -- scientific phase"),
    # Both reviewers share the dashed border + diagonal hatch convention:
    # they re-check an actor's output with an information barrier from the actor.
    mpatches.Patch(facecolor=FILL["expert"], edgecolor=CRITIC_EDGE,
                   linestyle="--", hatch=CRITIC_HATCH,
                   label="LITREVIEW critic -- info-barriered review (scientific)"),
    mpatches.Patch(facecolor=VALIDATOR_FILL, edgecolor=EDGE["dataml"],
                   linestyle="--", hatch=CRITIC_HATCH,
                   label="DATAML validator -- info-barriered review (mechanical)"),
]
ax.legend(handles=legend_items, loc="lower center",
          bbox_to_anchor=(0.5, -0.14), ncol=3,
          frameon=False, fontsize=8.5, handlelength=2.0, handleheight=1.1)

ax.set_title(
    "Expert Review Pipeline v29 -- actor / reviewer separation across 21 phases (+ 20a)",
    fontsize=12.5, weight="bold", pad=10,
)

plt.subplots_adjust(left=0.02, right=0.99, top=0.92, bottom=0.16)

out_path = os.path.join(
    os.path.dirname(os.path.abspath("__nb__")), "..", "fig_methods_pipeline.png"
)
fig.savefig(out_path, dpi=200, bbox_inches="tight", facecolor="white")
print("saved:", out_path)
